# Kyiv Apartment Rent Price Prediction

**Course:** Introduction to Machine Learning  
**Project type:** Supervised learning / regression  
**Goal:** predict monthly apartment rent prices in Kyiv using property characteristics scraped from LUN.

This notebook is written as a reproducible final report: every step is commented and can be run from start to finish.

## 1. Problem Definition

The practical problem is to estimate apartment rental prices in Kyiv based on observable property characteristics. This is useful for tenants, landlords, real-estate platforms, and analysts who want to understand which factors are associated with higher or lower rent.

- **Prediction target:** monthly rent price in UAH.
- **Type of task:** regression.
- **Predictors:** apartment size, rooms, floor, district, construction technology, heating type, geographic coordinates, and other available characteristics.

Because rental prices are strongly right-skewed, the models are trained on `log_price`, but the final evaluation is also reported in UAH for easier interpretation.

In [ ]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# Machine learning
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

# Display options
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

RANDOM_STATE = 42

## 2. Data Loading

The dataset contains apartment rental listings from Kyiv. Each row corresponds to one listing. The raw file includes prices in several currencies, apartment characteristics, location, and description text.

In [ ]:
# If you use Google Colab, upload kyiv_flats_data.csv to the session first,
# or keep it in the same folder as this notebook.

DATA_PATH = "kyiv_flats_data.csv"
raw = pd.read_csv(DATA_PATH)

print("Raw data shape:", raw.shape)
raw.head()

In [ ]:
raw.info()

In [ ]:
# Missing values in raw data
missing_raw = raw.isna().sum().sort_values(ascending=False)
missing_raw

## 3. Data Cleaning and Feature Engineering

The raw dataset requires several cleaning steps:

1. Remove duplicate listings by URL.
2. Remove observations with missing target values.
3. Convert all prices to UAH.
4. Convert numeric-looking columns stored as text into numeric format.
5. Create additional features such as building age and floor ratio.
6. Remove extreme or unrealistic outliers.

Currency conversion is based on fixed approximate exchange rates used for reproducibility. If the project is updated later, these rates can be replaced with official NBU rates for the exact scraping date.

In [ ]:
df = raw.copy()

# 1. Remove duplicated listings
initial_rows = len(df)
df = df.drop_duplicates(subset=["url"])
print("Removed duplicated URLs:", initial_rows - len(df))

# 2. Drop rows where target or key features are missing
df = df.dropna(subset=["price", "currency", "area_total", "rooms"])

# 3. Convert all prices into UAH
# Fixed approximate rates for reproducibility. Update if needed.
exchange_rates = {
    "UAH": 1.0,
    "USD": 41.5,
    "EUR": 45.0
}

df["price_uah"] = df["price"] * df["currency"].map(exchange_rates)
df = df.dropna(subset=["price_uah"])

# 4. Convert numeric columns that were stored as text
for col in ["area_living", "area_kitchen", "floor"]:
    df[col] = pd.to_numeric(df[col].replace("-", np.nan), errors="coerce")

# 5. Feature engineering
# Commission is often missing. We use a simple indicator: whether commission information exists.
df["has_commission"] = df["commission"].notna().astype(int)

# Building age. Unrealistic values are treated as missing.
df["building_age"] = 2026 - df["build_year"]
df.loc[(df["building_age"] < 0) | (df["building_age"] > 200), "building_age"] = np.nan

# Relative floor position in the building
df["floor_ratio"] = df["floor"] / df["total_floors"]
df.loc[(df["floor_ratio"] <= 0) | (df["floor_ratio"] > 1.5), "floor_ratio"] = np.nan

print("Shape after basic cleaning:", df.shape)
df[["price", "currency", "price_uah", "rooms", "area_total", "floor", "total_floors"]].head()

In [ ]:
# 6. Outlier treatment
# We remove observations that are clearly outside a reasonable rental-apartment range.
# These thresholds are transparent and can be discussed as a limitation.

before = len(df)

df = df[
    (df["price_uah"] >= 3_000) &
    (df["price_uah"] <= 500_000) &
    (df["area_total"] >= 10) &
    (df["area_total"] <= 400) &
    (df["rooms"] >= 1) &
    (df["rooms"] <= 10) &
    (df["total_floors"] >= 1) &
    (df["total_floors"] <= 100)
].copy()

print("Rows removed as outliers:", before - len(df))
print("Cleaned data shape:", df.shape)

In [ ]:
# Final target variable
# We model log(price) because real-estate prices are usually right-skewed.
df["log_price"] = np.log(df["price_uah"])

df[["price_uah", "log_price", "rooms", "area_total", "district"]].describe(include="all")

## 4. Exploratory Data Analysis

The goal of EDA is to understand the target distribution, key predictors, outliers, and potential relationships between variables.

In [ ]:
# Price distribution in UAH
plt.figure(figsize=(8, 5))
plt.hist(df["price_uah"], bins=60)
plt.title("Distribution of Monthly Rent Prices in Kyiv")
plt.xlabel("Monthly rent, UAH")
plt.ylabel("Number of listings")
plt.show()

In [ ]:
# Log price distribution
plt.figure(figsize=(8, 5))
plt.hist(df["log_price"], bins=60)
plt.title("Distribution of Log Monthly Rent Prices")
plt.xlabel("log(price_uah)")
plt.ylabel("Number of listings")
plt.show()

In [ ]:
# Relationship between area and price
plt.figure(figsize=(8, 5))
plt.scatter(df["area_total"], df["price_uah"], alpha=0.25)
plt.title("Rent Price vs Total Area")
plt.xlabel("Total area, m²")
plt.ylabel("Monthly rent, UAH")
plt.show()

In [ ]:
# Average rent by district
avg_by_district = (
    df.groupby("district", dropna=False)["price_uah"]
    .agg(["count", "mean", "median"])
    .sort_values("median", ascending=False)
)

avg_by_district.head(15)

In [ ]:
# Visualize districts with enough observations
plot_districts = avg_by_district[avg_by_district["count"] >= 50].sort_values("median")

plt.figure(figsize=(8, 6))
plt.barh(plot_districts.index.astype(str), plot_districts["median"])
plt.title("Median Monthly Rent by District")
plt.xlabel("Median rent, UAH")
plt.ylabel("District")
plt.show()

In [ ]:
# Price by number of rooms
rooms_summary = df.groupby("rooms")["price_uah"].agg(["count", "mean", "median"]).sort_index()
rooms_summary

In [ ]:
plt.figure(figsize=(8, 5))
plt.boxplot(
    [df.loc[df["rooms"] == r, "price_uah"] for r in sorted(df["rooms"].dropna().unique())],
    labels=[str(int(r)) for r in sorted(df["rooms"].dropna().unique())],
    showfliers=False
)
plt.title("Rent Price by Number of Rooms")
plt.xlabel("Number of rooms")
plt.ylabel("Monthly rent, UAH")
plt.show()

In [ ]:
# Correlation between numeric variables
numeric_cols = [
    "price_uah", "log_price", "rooms", "area_total", "area_living", "area_kitchen",
    "floor", "total_floors", "building_age", "floor_ratio", "lat", "lon", "has_commission"
]

corr = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

### EDA interpretation

The rent-price distribution is right-skewed: most listings have moderate prices, while a smaller number of premium apartments are much more expensive. This is why `log_price` is used as the modeling target. Larger apartments and apartments in more central or expensive districts tend to have higher rents. Location, area, and number of rooms are expected to be among the most important predictors.

## 5. Model Preparation

We use the following predictors. Numeric variables are imputed with the median and standardized. Categorical variables are imputed with the most frequent category and one-hot encoded.

The train-test split is 80/20. The models are trained on `log_price`, while final metrics are reported both on the log scale and after converting predictions back to UAH.

In [ ]:
features = [
    "rooms", "area_total", "area_living", "area_kitchen", "floor", "total_floors",
    "building_age", "floor_ratio", "lat", "lon", "district", "construction_tech",
    "heating_type", "has_commission"
]

target = "log_price"

model_data = df[features + ["price_uah", target]].copy()

X = model_data[features]
y = model_data[target]

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Modeling data shape:", model_data.shape)

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

## 6. Model Development

We estimate four models:

1. **Linear Regression** — simple interpretable baseline.
2. **Ridge Regression** — linear model with L2 regularization.
3. **Random Forest** — advanced nonlinear model based on many decision trees.
4. **Gradient Boosting** — advanced nonlinear ensemble model.

This allows us to compare interpretability and predictive accuracy.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": RidgeCV(alphas=[0.1, 1, 10, 50, 100]),
    "Random Forest": RandomForestRegressor(
        n_estimators=150,
        max_depth=18,
        min_samples_leaf=3,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3,
        random_state=RANDOM_STATE
    )
}

In [ ]:
def evaluate_model(name, pipeline, X_train, X_test, y_train, y_test):
    """Fit model and compute metrics on train and test data."""
    pipeline.fit(X_train, y_train)
    
    train_pred_log = pipeline.predict(X_train)
    test_pred_log = pipeline.predict(X_test)
    
    # Convert log predictions back to UAH for practical interpretation
    train_pred_uah = np.exp(train_pred_log)
    test_pred_uah = np.exp(test_pred_log)
    train_true_uah = np.exp(y_train)
    test_true_uah = np.exp(y_test)
    
    return {
        "Model": name,
        "Train RMSE (UAH)": np.sqrt(mean_squared_error(train_true_uah, train_pred_uah)),
        "Test RMSE (UAH)": np.sqrt(mean_squared_error(test_true_uah, test_pred_uah)),
        "Train MAE (UAH)": mean_absolute_error(train_true_uah, train_pred_uah),
        "Test MAE (UAH)": mean_absolute_error(test_true_uah, test_pred_uah),
        "Train R2 (log)": r2_score(y_train, train_pred_log),
        "Test R2 (log)": r2_score(y_test, test_pred_log),
        "pipeline": pipeline
    }

In [ ]:
results = []
fitted_pipelines = {}

for name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", model)
    ])
    
    output = evaluate_model(name, pipe, X_train, X_test, y_train, y_test)
    fitted_pipelines[name] = output.pop("pipeline")
    results.append(output)

results_df = pd.DataFrame(results).sort_values("Test RMSE (UAH)")
results_df

## 7. Cross-Validation

To obtain a more stable estimate of predictive performance, we also run 5-fold cross-validation on the training set. The scoring is based on RMSE of log prices. Lower values are better.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = []

for name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", model)
    ])
    
    neg_mse_scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        scoring="neg_mean_squared_error",
        cv=cv,
        n_jobs=-1
    )
    
    rmse_scores = np.sqrt(-neg_mse_scores)
    cv_results.append({
        "Model": name,
        "CV RMSE log mean": rmse_scores.mean(),
        "CV RMSE log std": rmse_scores.std()
    })

cv_results_df = pd.DataFrame(cv_results).sort_values("CV RMSE log mean")
cv_results_df

## 8. Model Interpretation

For the linear model, coefficients are directly interpretable after preprocessing, but one-hot encoding makes the full coefficient table long. For the advanced models, we use permutation importance. It measures how much model performance worsens when each variable is randomly shuffled. If shuffling a variable strongly worsens performance, that variable is important.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_pipeline = fitted_pipelines[best_model_name]

print("Best model based on test RMSE:", best_model_name)

In [ ]:
# Permutation importance for the best model
# We use the original columns before preprocessing; this gives importance by original feature.
perm = permutation_importance(
    best_pipeline,
    X_test,
    y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    scoring="r2"
)

importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

importance_df

In [ ]:
top_importance = importance_df.head(10).sort_values("importance_mean")

plt.figure(figsize=(8, 5))
plt.barh(top_importance["feature"], top_importance["importance_mean"])
plt.title(f"Top 10 Feature Importances: {best_model_name}")
plt.xlabel("Permutation importance, decrease in R²")
plt.ylabel("Feature")
plt.show()

### Interpretation of model results

The best model is selected based on the lowest test RMSE. If Random Forest or Gradient Boosting performs better than Linear Regression, this suggests that Kyiv rent prices depend on nonlinear relationships and interactions between apartment characteristics. For example, the effect of area may differ by district or building type.

At the same time, Linear Regression and Ridge Regression remain useful as interpretable baselines. They provide a transparent benchmark and help us understand whether more complex models are actually needed.

## 9. Prediction Example

The next cell shows how to generate predictions for several test observations. This is useful for checking whether predicted prices are realistic.

In [ ]:
example_predictions = X_test.copy().head(10)
example_true = np.exp(y_test.head(10))
example_pred = np.exp(best_pipeline.predict(example_predictions))

prediction_check = example_predictions.copy()
prediction_check["actual_price_uah"] = example_true.values
prediction_check["predicted_price_uah"] = example_pred
prediction_check["prediction_error_uah"] = prediction_check["predicted_price_uah"] - prediction_check["actual_price_uah"]

prediction_check[["actual_price_uah", "predicted_price_uah", "prediction_error_uah", "rooms", "area_total", "district"]]

## 10. Discussion and Limitations

Main limitations:

1. **Exchange-rate assumption:** USD and EUR listings were converted to UAH using fixed approximate rates. More precise work should use official exchange rates for the scraping date.
2. **Listing prices are asking prices:** The dataset reflects advertised rents, not necessarily final transaction prices.
3. **Missing values:** Some variables have many missing observations, especially commission, building year, and district.
4. **Text information is mostly unused:** Descriptions may contain important information such as renovation quality, pets allowed, parking, appliances, blackout autonomy, and proximity to metro. This could be added through NLP feature engineering.
5. **Potential duplicates or repeated listings:** Even after removing duplicate URLs, very similar listings may remain.

These limitations should be mentioned in the final presentation and report.

## 11. Conclusions

This project applies supervised machine learning to predict monthly apartment rental prices in Kyiv. The target variable is log monthly rent in UAH. The models use apartment characteristics, location, and building information.

The project compares interpretable linear models with more flexible machine learning models. The final model should be chosen based on test performance and practical interpretability. In most real-estate datasets, ensemble tree-based models such as Random Forest or Gradient Boosting are expected to perform better because they capture nonlinear relationships and interactions.

The most important practical factors are expected to include apartment area, district/location, number of rooms, floor characteristics, and building-related variables.

## 12. Contribution Statement Template

Replace this section with your actual team contribution statement.

- **Student 1:** data collection, data cleaning, EDA, notebook documentation.
- **Student 2:** model development, model evaluation, interpretation, presentation preparation.

Both team members reviewed the final notebook, results, and presentation.